In [1]:
from pprint import pprint
import datasets as datasets_lib
import grain
import pandas as pd
import os
import fsspec
import numpy as np

import transformers
from tunix.generate import mappings

Dataset = datasets_lib.Dataset
AutoTokenizer = transformers.AutoTokenizer

from absl import logging as absl_logging

import logging
import sys

# ====== Logging Configuration ======
# 1. Force absl to use python logging
absl_logging.use_python_logging()

# 2. Configure the root logger
logging.basicConfig(
    stream=sys.stdout,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - [%(name)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)

# 3. Explicitly set levels for relevant loggers
logging.getLogger().setLevel(logging.INFO)
logging.getLogger("absl").setLevel(logging.INFO)

# 4. Set absl verbosity
absl_logging.set_verbosity(absl_logging.INFO)
absl_logging.set_stderrthreshold("info")

print("Logging configured at INFO level.")

/home/linchai_google_com/miniconda3/envs/tunix/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Logging configured at INFO level.


In [2]:
try:
  from GOOGLE_INTERNAL_PACKAGE_PATH.pyglib import gfile
  from etils import ecolab

  cm = ecolab.adhoc(
      source=ecolab.FROM_NOTEBOOK_OR_HEAD,
      reload="tunix",
      behavior="preferred",
      cell_autoreload=True,
  )

  file_open = gfile.Open

  NOTEBOOK_ENV = "g3"
except Exception:
  NOTEBOOK_ENV = "git"

  import contextlib
  cm = contextlib.nullcontext()

  file_open = fsspec.open


In [3]:

with cm:
  from tunix.models.qwen2 import model as qwen2_lib
  from tunix.models.qwen2 import params as qwen2_params_lib
  from tunix.generate import sampler as sampler_lib
  from tunix.utils import math_utils
# %%
from typing import Any, Dict, Optional
import jax
from jax import numpy as jnp
from flax import nnx
import orbax.checkpoint as ocp
from tqdm.auto import tqdm
import re

In [4]:
MODEL_VERSION = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
MODEL_PATH = "/mnt/disks/linchai-data/huggingface/hub/models--deepseek-ai--DeepSeek-R1-Distill-Qwen-1.5B/snapshots/ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562"
model_config = qwen2_lib.ModelConfig.deepseek_r1_distill_qwen_1p5b()


In [5]:
import os
os.environ["TPU_LOG_DIR"] = "~/my_tpu_logs"
os.environ["SKIP_JAX_PRECOMPILE"] = "True"

In [6]:


tokenizer = AutoTokenizer.from_pretrained(MODEL_VERSION)

2026-05-08 03:03:28 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"


2026-05-08 03:03:28 - WARNING - [huggingface_hub.utils._http] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-08 03:03:28 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562/config.json "HTTP/1.1 200 OK"


2026-05-08 03:03:28 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 03:03:28 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-08 03:03:28 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-08 03:03:28 - INFO - [httpx] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/ad9f0ae0864d7fbcd1cd905e3c6c5b069cc8b562/tokenizer_config.json "HTTP/1.1 200 OK"
2026-05-08 03:03:28 - INFO - [httpx] HTTP Request: GET https://huggingface.co/api/models/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B/tree/main/additional_chat_templates

In [7]:

trainer_mesh = jax.sharding.Mesh(np.array(jax.devices()[2:4]).reshape(1, 2), axis_names=["fsdp", "tp"])
rollout_mesh = jax.sharding.Mesh(np.array(jax.devices()[:2]).reshape(1, 2), axis_names=["fsdp", "tp"])


Could not access ~/my_tpu_logs: not found


In [8]:

print("Loading model from safe tensors...")
reference_mesh = rollout_mesh
with trainer_mesh:
  trainer_model = qwen2_params_lib.create_model_from_safe_tensors(file_dir=MODEL_PATH, config=model_config, mesh=trainer_mesh)
  ref_model = qwen2_params_lib.create_model_from_safe_tensors(file_dir=MODEL_PATH, config=model_config, mesh=reference_mesh)

import jax
jax.clear_caches()

Loading model from safe tensors...


In [9]:
from tunix.generate import vllm_sampler   # pylint: disable=g-import-not-at-top

mapping_config = mappings.MappingConfig()

INFO 05-08 03:03:47 [__init__.py:59] TPU info: node_name=maxtext-single-host-1-v5p-8 | tpu_type=v5p-8 | worker_id=0 | num_chips=4 | num_cores_per_chip=2
2026-05-08 03:03:47 - WARNING - [absl] Tensorflow library not found, tensorflow.io.gfile operations will use native shim calls. GCS paths (i.e. 'gs://...') cannot be accessed.


/home/linchai_google_com/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/home/linchai_google_com/miniconda3/envs/tunix/lib/python3.12/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


In [10]:
sampler_vllm = vllm_sampler.VllmSampler(
    tokenizer=tokenizer,
    config=vllm_sampler.VllmConfig(
        mesh=rollout_mesh,
        hbm_utilization=0.78,
        enable_dp_attention=True,
        init_with_random_weights=False,
        mapping_config=mappings.MappingConfig(),
        engine_kwargs={
            "model": MODEL_VERSION,
            "max_model_len": 4096,
            "max_num_seqs": 1,
            "max_num_batched_tokens": 4096,
        },
        return_logprobs=True,
    ),
)

2026-05-08 03:04:13 - INFO - [absl] Engine kwargs setting key 'model' with value 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'.
2026-05-08 03:04:13 - INFO - [absl] Engine kwargs setting key 'max_model_len' with value '4096'.
2026-05-08 03:04:13 - INFO - [absl] Engine kwargs setting key 'max_num_seqs' with value '1'.
2026-05-08 03:04:13 - INFO - [absl] Engine kwargs setting key 'max_num_batched_tokens' with value '4096'.
INFO 05-08 03:04:13 [attention_interface.py:53] Using default RPA kernel
INFO 05-08 03:04:13 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 05-08 03:04:13 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.
WARNING 05-08 03:04:13 [interface.py:240] Failed to import from vllm._C: ModuleNotFoundError("No module named 'vllm._C'")
WARNING 05-08 03:04:13 [interface.py:240] Failed to import from vllm._C: ModuleNotFoundError("No module nam

2026-05-08 03:04:15,871	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 05-08 03:04:15 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=4096.
INFO 05-08 03:04:15 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-08 03:04:15 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
INFO 05-08 03:04:15 [tpu_platform.py:190] Initialized sharding configuration: ShardingConfigManager(total_devices=2, sharding_strategy=ShardingStrategy(tensor_parallelism=2, expert_parallelism=1, sequence_parallelism=1, data_parallelism=1, attention_data_parallelism=1, attention_data_expert_parallelism=1, decode_context_parallelism=1), device_indexes=[0, 1])
INFO 05-08 03:04:15 [tpu_platform.py:247] Using KV cache block size: 256
INFO 05-08 03:04:15 [tpu_platform.py:254] TPU_MAMBA_SSM_CACHE_DTYPE=bfloat16 overriding cache_config.mamba_ssm_cache_dtype (was 'auto')
INFO 05-08 03:04:15 [tpu_platform.py:268] Force using UniProcExecutor for JAX on single host without pipeline parallelism.
INFO 05

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
W0508 03:04:23.283076 2694001 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.
E0508 03:04:23.283681 2695217 cpu_aot_loader.cc:220] Loading XLA:CPU AOT result. Target machine feature +prefer-no-gather is not  supported on the host machine. Machine type used for XLA:CPU compilation doesn't match the machine type for execution. Compile machine features: [+64bit,+adx,+aes,+amx-bf16,+amx-int8,+amx-tile,+avx,+avx2,+avx512bf16,+avx512bitalg,+avx512bw,+avx512cd,+avx512dq,+avx512f,+avx512fp16,+avx512ifma,+avx512vbmi,+avx512vbmi2,+avx512vl,+avx512vnni,+avx512vpopcntdq,+avxvnni,+bmi,+bmi2,+cldemote,+clflushopt,+clwb,+cmov,+crc32,+cx16,+cx8,+f16c,+fma,+fsgsbase,+fxsr,+gfni,+invpcid,+lzcnt,+mmx,+movbe,+movdir64b,+movdiri,+pclmul,+popcnt,+prefer-no-gather,+prefer-no-scatter,+prfchw,+rdpid,+rdrnd,+rdseed,+rtm,+sahf,+serialize,+sha,+sse,+sse2,+sse3,+sse4.1,+sse4.2,+ssse3

INFO 05-08 03:04:52 [default_loader.py:391] Loading weights took 30.64 seconds


INFO 05-08 03:04:53 [tpu_runner.py:616] Init model | hbm=[(3.53, 95.74), (3.53, 95.74)]GiB
INFO 05-08 03:04:53 [tpu_worker.py:380] Memory statistics | total_hbm_limit_gb=191.49GiB | total_hbm_limit_cap_gb=149.36GiB | total_hbm_used_gb=7.06GiB | total_hbm_avail_gb=142.3GiB
INFO 05-08 03:04:53 [kv_cache_utils.py:1710] GPU KV cache size: 5,328,896 tokens
INFO 05-08 03:04:53 [kv_cache_utils.py:1711] Maximum concurrency for 4,096 tokens per request: 1301.00x
INFO 05-08 03:04:56 [kv_cache_manager.py:834] Hybrid KV cache layout: num_kv_cache_groups=1, num_kv_cache_tensors=28, kv_cache_config.num_blocks=20816, duplicate_shared_layers=False
INFO 05-08 03:04:56 [kv_cache_manager.py:862] Init kv-cache | num_total_layers=28 | num_blocks=[20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816, 20816] | regular_attn_layers=28 | regular_attn_shape=(num_blocks, (256, 2, 

In [12]:
jax.clear_caches()
prompts = [{'role': 'system', 'content': "You are a helpful assistant. You are walking on a frozen lake.\n\nFrozenLake Quick Guide\nGoal: Reach the goal (G). Player (P) and Goal (G) must overlap.\n\nSymbols:\n_ Frozen | O Hole | G Goal | P Player\n\nRules:\n1. Avoid falling into holes (O).\n2. Frozen tiles are slippery, you may move perpendicular to your intended direction.\n\nValid Action (separated by | ):\nUp | Down | Left | Right\n\nRewards:\nFall into hole: 0\nReach goal: +1.0\n\nYou will be provided the current observation, please decide on the next Action.\nYou should show your thought process and then input the final action in ``` ```.\nYou should only output the NEXT ACTION at each interation in the ``` ```. For example, if you want to move up, you should output ```Up```.\nYou should plan ahead and need to achieve it in minimum number of steps.\n\nBelow are examples for an interaction:\nExample1:\nUser: Current Observation:\nP   _   _   _   _\nO   _   _   O   _\nO   _   O   _   _\nO   _   _   G   _\n_   _   _   _   _\nYou have not achieved the goal, P has not reached G yet. Please give the next action.\n\nAssistant: P is now at the top right corner. It should reach G at the bottom right corner. I should move it closer to it. I can move right or down but there is a hole in down position and I can not move diagonally. There is no hole in my next movement right so I can move to right. Action: ```Right```\n\nExample2:\nUser: Current Observation:\n_   _   _   _\n_   _   _   O\n_   O   _   P\nO   _   _   G\nYou have not achieved the goal, P has not reached G yet. Please give the next action.\n\nAssistant: P is now at the near G. It should reach G to its bottom. I should move to be on it. There is no hole in my next movement so I can move to down. Action: ```Down```\n\nExample3:\nUser: Current Observation:\n_   _   _   O   _\nO   _   P   O   _\nO   _   O   _   _\nO   _   _   G   _\n_   _   _   _   _\nYou have not achieved the goal, P has not reached G yet. Please give the next action.\n\nAssistant: G is at the bottom right relative to P. I want to move closer so I should move right or down. But there is a hole at each position and I do not want to fall into holes. Up and left are both valid but left brings me closer. Action: ```Left```\n\nExample4:\nUser: Current Observation:\n_   _   _   _\n_   _   _   O\n_   O   _   O\nO   G   P   _\nYou have not achieved the goal, P has not reached G yet. Please give the next action.\n\nAssistant: P is now near G. But game has not finished. P is not at G and I should never output invalid action. I need to recheck my understanding. P is not actually on G yet because they are not overlapping, it needs reach G to its left. Action: ```Left```\n\nExample5:\nUser: Current Observation:\n_   _   _   O   _\nO   _   P   _   _\nO   _   O   O   O\nO   _   O   G   _\nO   _   _   _   _\nYou have not achieved the goal, P has not reached G yet. Please give the next action.\n\nAssistant: G is at the bottom right corner of P. I can move left, right, or up. Move right will initially bring me closer but I can't reach G that way. Move up and left means I can still reach G. Move up will result in 9 steps in total while left is 7 steps. I need to move left. Action: ```Left```\n\nNow it is your turn, please show your thinking process and put the final action in ``` ```. In every turn, the final action MUST be one of Up, Down, Left, Right.\n"}, {'role': 'user', 'content': 'Current Observation: \n G \t _ \t\n P \t _ \t\nYou have not achieved the goal, P has not reached G yet. Please give the next action.'}, {'role': 'assistant', 'content': 'P is currently at the bottom-left position. The goal G is directly above P. To reach the goal in the minimum number of steps, I should move up.\n\nAction: ```Up```<turn|>'}, {'role': 'user', 'content': 'Current Observation: \n √ \t _ \t\n _ \t _ \t'}]
prompts = tokenizer.apply_chat_template(prompts, add_generation_prompt=True, tokenize=False)
out_data = sampler_vllm(
    input_strings=prompts,
    max_generation_steps=16,
    max_prompt_length=2048,
    temperature=0,
    top_p=1,
    top_k=None,
    seed=None,
    echo=False,
    pad_output=False,
)
print("Text from VLLM sampler: ", out_data.text)
print("logprobs from VLLM sampler: ", out_data.logprobs)
print("tokens from VLLM sampler: ", out_data.tokens)
print("padded_prompt_tokens from VLLM sampler: ", out_data.padded_prompt_tokens)

WARNING 05-08 03:07:25 [model.py:1449] Default vLLM sampling parameters have been overridden by the model's `generation_config.json`: `{'temperature': 0.6, 'top_p': 0.95}`. If this is not intended, please relaunch vLLM instance with `--generation-config vllm`.
type of input_strings:  <class 'list'>
type of each input string:  <class 'str'>


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 05-08 03:07:42 [tpu_runner.py:828] Should not schedule a request that does nothing!


Processed prompts: 100%|██████████| 1/1 [00:17<00:00, 17.23s/it, est. speed input: 57.59 toks/s, output: 0.93 toks/s]

Text from VLLM sampler:  ["Okay, so I'm trying to figure out what action the player should take next"]
logprobs from VLLM sampler:  [[-0.6948671936988831, -3.4570546176837524e-06, -0.5779691338539124, -0.09856143593788147, -0.21041327714920044, -0.028058866038918495, 0.0, -0.022322699427604675, -4.0291848563356325e-05, -0.6351783275604248, -0.17727741599082947, -0.6281154751777649, -0.6760327816009521, -0.06261083483695984, -0.0006673480966128409, -0.07835843414068222]]
tokens from VLLM sampler:  [array([32313,    11,   773,   358,  2776,  4460,   311,  7071,   700,
        1128,  1917,   279,  2781,  1265,  1896,  1790], dtype=int32)]
padded_prompt_tokens from VLLM sampler:  [[151643 151643 151643 ... 151645 151648    198]]


In [21]:

out_tokens = np.array([32313,    11,   773,   358,  2776,  4460,   311,  7071,   700, 1128,  1917,   279,  2781,  1265,  1896,  1790, 0, 0], dtype=np.int32)


prompt_tokens = out_data.padded_prompt_tokens[0]
completion_tokens = out_tokens
completion_mask = np.array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], dtype=np.int32)
print(f"{prompt_tokens=}")
print(f"{completion_tokens=}")
prompt_tokens = jnp.repeat(jnp.array([prompt_tokens]), 1, axis=0)
completion_tokens = jnp.repeat(jnp.array([completion_tokens]), 1, axis=0)
completion_mask = jnp.repeat(jnp.array([completion_mask]), 1, axis=0)


from tunix.rl import common
graphdef, state = nnx.split(ref_model)

jax.clear_caches()

ref_logps = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=prompt_tokens,
    completion_tokens=completion_tokens,
    pad_id=tokenizer.pad_token_id,
    eos_id=tokenizer.eos_token_id,
)

prompt_tokens=array([151643, 151643, 151643, ..., 151645, 151648,    198],
      shape=(2048,), dtype=int32)
completion_tokens=array([32313,    11,   773,   358,  2776,  4460,   311,  7071,   700,
        1128,  1917,   279,  2781,  1265,  1896,  1790,     0,     0],
      dtype=int32)


W0508 03:20:37.709011 2694001 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.


In [22]:
import jax
jax.clear_caches()
jax.block_until_ready(ref_logps)
print(f"{ref_logps[0] = }")

# float32 [-1.6212287e-05, -4.2914307e-05,  0.0000000e+00, -1.2116224e-03,  0.0000000e+00,  0.0000000e+00, -1.1920902e-07, -5.9604508e-07]
# bf16 [-1.6212287e-05, -4.1364681e-05,  0.0000000e+00, -1.2746054e-03, 0.0000000e+00,  0.0000000e+00, -1.1920902e-07, -4.7683608e-07]
# vllm sampler: [-1.8954058759845793e-05, -3.337797534186393e-05, 0.0, -0.0013262758729979396, 0.0, 0.0, -1.1920901954454166e-07, -4.7683607817816664e-07]

ref_logps[0] = Array([-4.4388968e-01, -5.6028093e-06, -4.2856404e-01, -1.8758525e-01,
       -5.8636885e-02, -1.6544402e-01, -5.9604508e-07, -3.6298331e-02,
       -8.0701306e-05, -1.1772163e+00, -4.0633923e-01, -1.3259853e+00,
       -2.5309697e-01, -9.8135933e-02, -9.3213876e-04, -9.6031561e-02,
       -1.6899717e+01, -2.0486887e+01], dtype=float32)


In [23]:

from tunix.rl import common
graphdef, state = nnx.split(trainer_model)

jax.clear_caches()

trainer_logps = common.compute_per_token_logps(
    graphdef,
    state,
    prompt_tokens=prompt_tokens,
    completion_tokens=completion_tokens,
    completion_mask=completion_mask,
    pad_id=tokenizer.pad_token_id,
    eos_id=tokenizer.eos_token_id,
)

In [24]:
trainer_logps = jax.device_get(trainer_logps)
print(f"{trainer_logps[0] = }")

trainer_logps[0] = array([-4.4388968e-01, -5.6028093e-06, -4.2856404e-01, -1.8758525e-01,
       -5.8636885e-02, -1.6544402e-01, -5.9604508e-07, -3.6298331e-02,
       -8.0701306e-05, -1.1772163e+00, -4.0633923e-01, -1.3259853e+00,
       -2.5309697e-01, -9.8135933e-02, -9.3213876e-04, -9.6031561e-02,
       -1.6899717e+01, -1.6448734e+01], dtype=float32)


In [25]:
kl = common.compute_kl_divergence(
        trainer_logps,
        ref_logps,
        "low_var_kl",
    )

In [26]:
kl

Array([[0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
        0.       , 0.       , 0.       , 0.       , 0.       , 3.0557828]],      dtype=float32)